<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Amortization_Exp1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================================
# EXPERIMENT 5.1-B — SINGLE-REPLICATION LEARNING CURVE
#
# TRAINING N:
#   30,50,70,90,110,130,150
#
# TEST:
#   exact N         = training grid
#   interpolation N = 40,60,80,100,120,140
#
# TRAINING-SAMPLE SIZES:
#   R = 100,...,50000
#
# SINGLE REPLICATION ONLY
#
# LOSS:
#   L = L_PMF + lambda_rho L_tail + lambda_tau L_tau
#
# R0 = beta/gamma
#
# SPARSE-LU ONLY — NO MATRIX INVERSE
# =====================================================================================

from __future__ import annotations

import copy
import hashlib
import json
import math
import pickle
import random
import time

from dataclasses import dataclass, asdict
from functools import lru_cache
from pathlib import Path
from typing import Tuple

import numpy as np

from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.stats import qmc

import torch
import torch.nn as nn

import matplotlib.pyplot as plt


# =====================================================================================
# 1. CONFIG
# =====================================================================================

@dataclass
class Config:

    seed: int = 20260820

    beta_range: Tuple[float, float] = (.30, 1.50)
    gamma_range: Tuple[float, float] = (.20, 1.00)
    omega_range: Tuple[float, float] = (.02, .50)

    initial_fraction_range: Tuple[float, float] = (.02, .20)
    i0_one_fraction: float = .25

    train_N: Tuple[int, ...] = (
        30, 50, 70, 90, 110, 130, 150
    )

    interp_N: Tuple[int, ...] = (
        40, 60, 80, 100, 120, 140
    )

    n_train: int = 50000
    n_val: int = 500

    n_test_seen: int = 500
    n_test_interp: int = 450

    width: int = 128
    depth: int = 3
    batch_size: int = 64

    epochs: int = 500

    lr: float = 1e-3
    weight_decay: float = 1e-6

    lambda_rho: float = 1.0
    lambda_tau: float = .02

    patience: int = 50
    min_delta: float = 1e-6
    grad_clip: float = 5.

    prob_tol: float = 1e-10
    var_rel_tol: float = 1e-8
    kl_eps: float = 1e-12
    refine_steps: int = 3

    teacher_version: str = (
        "B_R50000_N150_tailaware_single_rep_v14"
    )

    output_dir: str = (
        "results_section_5_1B_R50000_single_rep"
    )

    dpi: int = 300


cfg = Config()


R_VALUES = (
    100,
    200,
    500,
    1000,
    2000,
    5000,
    10000,
    20000,
    30000,
    50000
)


GALLERY_R = (
    100,
    500,
    2000,
    10000,
    50000
)


EXACT_TEST_N = cfg.train_N
INTERP_TEST_N = cfg.interp_N

ALL_TEST_N = tuple(
    sorted(
        set(
            EXACT_TEST_N
            +
            INTERP_TEST_N
        )
    )
)


REPLICATION_ID = 0


assert set(
    EXACT_TEST_N
).isdisjoint(
    INTERP_TEST_N
)

assert cfg.n_train >= max(
    R_VALUES
)


def seed_all(s):

    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


seed_all(
    cfg.seed
)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else
    "cpu"
)


out = Path(
    cfg.output_dir
)

out.mkdir(
    parents=True,
    exist_ok=True
)


tag = hashlib.sha1(
    json.dumps(
        asdict(cfg),
        sort_keys=True
    ).encode()
).hexdigest()[:12]


N_scale = max(
    cfg.train_N
)


print("=" * 115)
print("EXPERIMENT 5.1-B — SINGLE-REPLICATION LEARNING CURVE")
print("=" * 115)

print(
    "Device:",
    device
)

print(
    "Training N:",
    cfg.train_N
)

print(
    "Exact test N:",
    EXACT_TEST_N
)

print(
    "Interpolated test N:",
    INTERP_TEST_N
)

print(
    "R values:",
    R_VALUES
)

print(
    "Replications: 1"
)

print(
    "R0 = beta/gamma"
)

print(
    "No matrix inverse is computed."
)


# =====================================================================================
# 2. CACHED SIRS STATE-SPACE TOPOLOGY
# =====================================================================================

@lru_cache(
    maxsize=None
)
def topology(N):

    states = [
        (s, i)
        for i in range(1, N + 1)
        for s in range(N - i + 1)
    ]

    idx = {
        x: j
        for j, x in enumerate(states)
    }

    M = len(states)

    ir = []
    ic = []
    ib = []

    rr = []
    rc = []
    rb = []

    wr = []
    wc = []
    wb = []

    db = np.zeros(M)
    dg = np.zeros(M)
    dw = np.zeros(M)
    qb = np.zeros(M)

    for row, (s, i) in enumerate(states):

        r = N - s - i

        # Infection
        if s:

            ir.append(row)

            ic.append(
                idx[
                    (s - 1, i + 1)
                ]
            )

            ib.append(
                s * i / N
            )

            db[row] = (
                s * i / N
            )

        # Recovery rate always contributes
        # to the diagonal.
        dg[row] = i

        # If i=1 recovery goes directly
        # to the absorbing set I=0.
        if i == 1:

            qb[row] = i

        else:

            rr.append(row)

            rc.append(
                idx[
                    (s, i - 1)
                ]
            )

            rb.append(i)

        # Immunity loss
        if r:

            wr.append(row)

            wc.append(
                idx[
                    (s + 1, i)
                ]
            )

            wb.append(r)

            dw[row] = r

    return (
        idx,
        M,

        np.asarray(
            ir,
            dtype=int
        ),

        np.asarray(
            ic,
            dtype=int
        ),

        np.asarray(
            ib,
            dtype=float
        ),

        np.asarray(
            rr,
            dtype=int
        ),

        np.asarray(
            rc,
            dtype=int
        ),

        np.asarray(
            rb,
            dtype=float
        ),

        np.asarray(
            wr,
            dtype=int
        ),

        np.asarray(
            wc,
            dtype=int
        ),

        np.asarray(
            wb,
            dtype=float
        ),

        db,
        dg,
        dw,
        qb
    )


def matrices(
    beta,
    gamma,
    omega,
    N
):

    (
        idx,
        M,

        ir,
        ic,
        ib,

        rr,
        rc,
        rb,

        wr,
        wc,
        wb,

        db,
        dg,
        dw,
        qb

    ) = topology(N)

    rows = np.concatenate([
        ir,
        rr,
        wr,
        np.arange(M)
    ])

    cols = np.concatenate([
        ic,
        rc,
        wc,
        np.arange(M)
    ])

    vals = np.concatenate([
        beta * ib,
        gamma * rb,
        omega * wb,
        -(
            beta * db
            +
            gamma * dg
            +
            omega * dw
        )
    ])

    T = sparse.coo_matrix(
        (
            vals,
            (rows, cols)
        ),
        shape=(M, M),
        dtype=np.float64
    ).tocsc()

    D1 = sparse.coo_matrix(
        (
            beta * ib,
            (ir, ic)
        ),
        shape=(M, M),
        dtype=np.float64
    ).tocsc()

    q = gamma * qb

    return (
        T,
        (T - D1).tocsc(),
        D1,
        q,
        idx
    )


# =====================================================================================
# 3. SPARSE LU SOLVES
# =====================================================================================

def factor(
    A,
    ordering="COLAMD"
):

    return splu(
        A.tocsc(),
        permc_spec=ordering
    )


def solve_lu(
    A,
    lu,
    b,
    transpose=False
):

    b = np.asarray(
        b,
        dtype=np.float64
    )

    mode = (
        "T"
        if transpose
        else
        "N"
    )

    x = lu.solve(
        b,
        trans=mode
    )

    for _ in range(
        cfg.refine_steps
    ):

        residual = (
            b - A.T @ x
            if transpose
            else
            b - A @ x
        )

        if not np.all(
            np.isfinite(
                residual
            )
        ):
            break

        rel = (
            np.linalg.norm(
                residual,
                np.inf
            )
            /
            max(
                np.linalg.norm(
                    b,
                    np.inf
                ),
                1.
            )
        )

        if rel < 1e-11:
            break

        x += lu.solve(
            residual,
            trans=mode
        )

    return np.asarray(
        x,
        dtype=np.float64
    )


def moment_factor(
    A,
    ordering
):

    d = np.abs(
        A.diagonal()
    )

    scale = 1. / np.maximum(
        d,
        np.finfo(float).tiny
    )

    As = (
        sparse.diags(
            scale
        )
        @
        A
    ).tocsc()

    lu = splu(
        As,
        permc_spec=ordering
    )

    return (
        lu,
        scale
    )


def moment_solve(
    A,
    lu,
    scale,
    b
):

    b = np.asarray(
        b,
        dtype=np.float64
    )

    x = lu.solve(
        scale * b
    )

    for _ in range(
        cfg.refine_steps
    ):

        residual = (
            b
            -
            A @ x
        )

        if not np.all(
            np.isfinite(
                residual
            )
        ):
            break

        rel = (
            np.linalg.norm(
                residual,
                np.inf
            )
            /
            max(
                np.linalg.norm(
                    b,
                    np.inf
                ),
                1.
            )
        )

        if rel < 1e-11:
            break

        x += lu.solve(
            scale
            *
            residual
        )

    return np.asarray(
        x,
        dtype=np.float64
    )


# =====================================================================================
# 4. EXTINCTION MOMENTS
# =====================================================================================

def variance_linear_system(
    T,
    q,
    A,
    lu,
    scale,
    m1
):

    C = T.tocoo()

    keep = (
        C.row
        !=
        C.col
    )

    rr = C.row[
        keep
    ]

    cc = C.col[
        keep
    ]

    rates = C.data[
        keep
    ]

    source = np.bincount(
        rr,
        weights=
            rates
            *
            (
                m1[cc]
                -
                m1[rr]
            )**2,
        minlength=T.shape[0]
    ).astype(
        np.float64
    )

    source += (
        q
        *
        m1
        *
        m1
    )

    if (
        not np.all(
            np.isfinite(
                source
            )
        )
        or
        source.min() < -1e-8
    ):

        raise ArithmeticError(
            "Invalid variance-system RHS."
        )

    return moment_solve(
        A,
        lu,
        scale,
        np.maximum(
            source,
            0.
        )
    )


def extinction_moments(
    T,
    q,
    initial
):

    A = (
        -T
    ).tocsc()

    one = np.ones(
        A.shape[0],
        dtype=np.float64
    )

    errors = []

    for ordering in (
        "COLAMD",
        "MMD_AT_PLUS_A"
    ):

        try:

            lu, scale = moment_factor(
                A,
                ordering
            )

            # ----------------------------------------------------
            # (-T) m1 = 1
            # ----------------------------------------------------

            m1 = moment_solve(
                A,
                lu,
                scale,
                one
            )

            mean = float(
                m1[
                    initial
                ]
            )

            if (
                not np.all(
                    np.isfinite(
                        m1
                    )
                )
                or
                mean <= 0
            ):

                raise ArithmeticError(
                    f"Invalid E(tau)={mean}"
                )

            # ----------------------------------------------------
            # (-T) m2 = 2 m1
            # ----------------------------------------------------

            m2 = moment_solve(
                A,
                lu,
                scale,
                2. * m1
            )

            second = float(
                m2[
                    initial
                ]
            )

            if (
                not np.all(
                    np.isfinite(
                        m2
                    )
                )
                or
                second <= 0
            ):

                raise ArithmeticError(
                    f"Invalid E(tau^2)={second}"
                )

            raw = (
                np.longdouble(
                    second
                )
                -
                np.longdouble(
                    mean
                )**2
            )

            tol = (
                cfg.var_rel_tol
                *
                max(
                    abs(second),
                    mean * mean,
                    1.
                )
            )

            if (
                np.isfinite(raw)
                and
                raw >= -tol
            ):

                var = max(
                    float(raw),
                    0.
                )

                method = (
                    "moment subtraction"
                )

            else:

                vv = variance_linear_system(
                    T,
                    q,
                    A,
                    lu,
                    scale,
                    m1
                )

                var = float(
                    vv[
                        initial
                    ]
                )

                method = (
                    "variance linear system"
                )

            if (
                not np.isfinite(
                    var
                )
                or
                var < 0
            ):

                raise ArithmeticError(
                    f"Invalid Var(tau)={var}"
                )

            return (
                mean,
                second,
                var,
                True,
                f"{ordering}; {method}"
            )

        except Exception as e:

            errors.append(
                f"{ordering}: {e}"
            )

    return (
        np.nan,
        np.nan,
        np.nan,
        False,
        " | ".join(
            errors
        )
    )


# =====================================================================================
# 5. EXACT TEACHER
# =====================================================================================

def exact_targets(
    beta,
    gamma,
    omega,
    N,
    i0
):

    (
        T,
        D0,
        D1,
        q,
        idx

    ) = matrices(
        beta,
        gamma,
        omega,
        N
    )

    M = T.shape[0]

    initial = idx[
        (
            N - i0,
            i0
        )
    ]

    # ------------------------------------------------------------
    # Infection-count distribution
    # ------------------------------------------------------------

    A0 = (
        -D0
    ).tocsc()

    lu0 = factor(
        A0
    )

    alpha = np.zeros(
        M
    )

    alpha[
        initial
    ] = 1.

    b = solve_lu(
        A0,
        lu0,
        q
    )

    v = alpha.copy()

    p = np.zeros(
        N + 2,
        dtype=np.float64
    )

    for k in range(
        N + 1
    ):

        p[k] = (
            v @ b
        )

        y = solve_lu(
            A0,
            lu0,
            v,
            transpose=True
        )

        v = np.asarray(
            D1.T @ y
        ).ravel()

    p[-1] = (
        v.sum()
    )

    p[
        np.abs(p)
        <
        cfg.prob_tol
    ] = 0.

    if (
        not np.all(
            np.isfinite(
                p
            )
        )
        or
        p.min()
        <
        -cfg.prob_tol
    ):

        raise RuntimeError(
            f"Invalid exact PMF: "
            f"N={N}, i0={i0}"
        )

    p = np.maximum(
        p,
        0.
    )

    mass = (
        p.sum()
    )

    if (
        not np.isfinite(
            mass
        )
        or
        abs(
            mass - 1.
        )
        >
        1e-5
    ):

        raise RuntimeError(
            f"Invalid PMF mass={mass}: "
            f"N={N}, i0={i0}"
        )

    p /= mass

    (
        mean,
        second,
        var,
        tau_valid,
        tau_info

    ) = extinction_moments(
        T,
        q,
        initial
    )

    return (
        p,
        mean,
        second,
        var,
        tau_valid,
        tau_info
    )


# =====================================================================================
# 6. DESIGN + DATA
# =====================================================================================

@dataclass
class Record:

    beta: float
    gamma: float
    omega: float

    N: int
    i0: int

    p: np.ndarray

    mean_tau: float
    second_tau: float
    var_tau: float

    tau_valid: bool
    tau_info: str


def stretch(
    u,
    bounds
):

    a, b = bounds

    return (
        a
        +
        (b - a)
        *
        u
    )


def design(
    n,
    Ns,
    seed
):

    U = qmc.LatinHypercube(
        d=4,
        seed=seed
    ).random(
        n
    )

    beta = stretch(
        U[:, 0],
        cfg.beta_range
    )

    gamma = stretch(
        U[:, 1],
        cfg.gamma_range
    )

    omega = stretch(
        U[:, 2],
        cfg.omega_range
    )

    frac = stretch(
        U[:, 3],
        cfg.initial_fraction_range
    )

    Nv = np.tile(
        np.asarray(
            Ns
        ),
        math.ceil(
            n
            /
            len(Ns)
        )
    )[:n]

    rng = (
        np.random.default_rng(
            seed + 991
        )
    )

    rng.shuffle(
        Nv
    )

    i0 = np.asarray([
        int(
            np.clip(
                round(
                    frac[j]
                    *
                    Nv[j]
                ),
                2,
                Nv[j]
            )
        )
        for j in range(n)
    ])

    for N in Ns:

        ix = np.where(
            Nv == N
        )[0]

        k = max(
            1,
            int(
                round(
                    cfg.i0_one_fraction
                    *
                    len(ix)
                )
            )
        )

        i0[
            rng.choice(
                ix,
                k,
                replace=False
            )
        ] = 1

    return [
        (
            float(
                beta[j]
            ),

            float(
                gamma[j]
            ),

            float(
                omega[j]
            ),

            int(
                Nv[j]
            ),

            int(
                i0[j]
            )
        )

        for j in range(n)
    ]


def make_dataset(
    configs,
    name
):

    ans = []

    unresolved = 0

    t0 = (
        time.perf_counter()
    )

    for j, (
        b,
        g,
        w,
        N,
        i0

    ) in enumerate(
        configs,
        1
    ):

        try:

            (
                p,
                m,
                m2,
                v,
                ok,
                info

            ) = exact_targets(
                b,
                g,
                w,
                N,
                i0
            )

        except Exception as e:

            raise RuntimeError(
                f"\nEXACT DISTRIBUTIONAL TEACHER FAILED\n"
                f"{name}, record {j}\n"
                f"N={N}, i0={i0}, "
                f"beta={b:.8g}, "
                f"gamma={g:.8g}, "
                f"omega={w:.8g}\n"
                f"{e}"
            ) from e

        unresolved += int(
            not ok
        )

        ans.append(
            Record(
                b,
                g,
                w,
                N,
                i0,
                p,
                m,
                m2,
                v,
                ok,
                info
            )
        )

        if (
            j % 25 == 0
            or
            j == len(configs)
        ):

            print(
                f"[{name:12s}] "
                f"{j:5d}/{len(configs):5d} | "
                f"N={N:3d}, i0={i0:3d} | "
                f"tau unresolved={unresolved:3d} | "
                f"{time.perf_counter()-t0:.1f}s"
            )

    return ans


def load_or_make(
    name,
    configs
):

    path = (
        out
        /
        f"{name}_{cfg.teacher_version}_{tag}.pkl"
    )

    if path.exists():

        print(
            "Loading",
            path
        )

        with open(
            path,
            "rb"
        ) as f:

            return pickle.load(
                f
            )

    x = make_dataset(
        configs,
        name
    )

    with open(
        path,
        "wb"
    ) as f:

        pickle.dump(
            x,
            f
        )

    return x


train = load_or_make(
    "train",
    design(
        cfg.n_train,
        cfg.train_N,
        cfg.seed + 1
    )
)


valid = load_or_make(
    "validation",
    design(
        cfg.n_val,
        cfg.train_N,
        cfg.seed + 2
    )
)


test_seen = load_or_make(
    "test_exact",
    design(
        cfg.n_test_seen,
        EXACT_TEST_N,
        cfg.seed + 3
    )
)


test_interp = load_or_make(
    "test_interp",
    design(
        cfg.n_test_interp,
        INTERP_TEST_N,
        cfg.seed + 4
    )
)


# =====================================================================================
# 7. AUDIT
# =====================================================================================

def audit(
    records,
    name
):

    bad = [
        r
        for r in records
        if (
            not np.all(
                np.isfinite(
                    r.p
                )
            )
            or
            abs(
                r.p.sum()
                -
                1.
            )
            >
            1e-6
        )
    ]

    if bad:

        raise RuntimeError(
            f"Invalid probability targets in {name}."
        )

    print(
        "\n"
        +
        name
    )

    for N in sorted({
        r.N
        for r in records
    }):

        x = [
            r
            for r in records
            if r.N == N
        ]

        print(
            f"N={N:3d} | "
            f"n={len(x):3d} | "
            f"i0=1={sum(r.i0==1 for r in x):3d} | "
            f"valid tau="
            f"{sum(r.tau_valid for r in x):3d}/{len(x):3d}"
        )


print(
    "\n"
    +
    "=" * 115
)

print(
    "EXACT-TARGET AUDIT"
)

print(
    "=" * 115
)


audit(
    train,
    "TRAIN"
)

audit(
    valid,
    "VALIDATION"
)

audit(
    test_seen,
    "TEST — EXACT N"
)

audit(
    test_interp,
    "TEST — INTERPOLATED N"
)


# =====================================================================================
# 8. NETWORKS
# =====================================================================================

def mlp(
    din,
    dout
):

    L = []

    d = din

    for _ in range(
        cfg.depth
    ):

        L += [
            nn.Linear(
                d,
                cfg.width
            ),

            nn.SiLU()
        ]

        d = (
            cfg.width
        )

    L.append(
        nn.Linear(
            d,
            dout
        )
    )

    return nn.Sequential(
        *L
    )


class HazardNet(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        self.net = mlp(
            6,
            1
        )

    def forward(
        self,
        x
    ):

        return torch.sigmoid(
            self.net(
                x
            ).squeeze(-1)
        )


class TauNet(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        self.net = mlp(
            5,
            2
        )

    def forward(
        self,
        x
    ):

        return torch.nn.functional.softplus(
            self.net(
                x
            )
        )


def xtau(r):

    return torch.tensor(
        [
            r.beta,
            r.gamma,
            r.omega,
            r.N / N_scale,
            r.i0 / r.N
        ],
        dtype=torch.float32,
        device=device
    )


def xhaz(r):

    c = torch.arange(
        r.N + 1,
        dtype=torch.float32,
        device=device
    )

    n = (
        r.N + 1
    )

    return torch.column_stack([

        torch.full(
            (n,),
            r.beta,
            device=device
        ),

        torch.full(
            (n,),
            r.gamma,
            device=device
        ),

        torch.full(
            (n,),
            r.omega,
            device=device
        ),

        torch.full(
            (n,),
            r.N / N_scale,
            device=device
        ),

        torch.full(
            (n,),
            r.i0 / r.N,
            device=device
        ),

        c / r.N
    ])


def reconstruct(h):

    before = torch.cat([

        torch.ones(
            1,
            device=h.device
        ),

        torch.cumprod(
            1 - h[:-1],
            0
        )
    ])

    return torch.cat([

        before * h,

        torch.prod(
            1 - h
        ).reshape(1)
    ])


def tail_torch(p):

    return torch.flip(
        torch.cumsum(
            torch.flip(
                p[1:],
                dims=[0]
            ),
            dim=0
        ),
        dims=[0]
    )


def phat_tensor(
    model,
    r
):

    return reconstruct(
        model(
            xhaz(r)
        )
    )


def tau_target(r):

    if not r.tau_valid:

        raise RuntimeError(
            "tau_target called for unresolved tau target."
        )

    raw = np.asarray(
        [
            r.mean_tau,
            r.var_tau
        ],
        dtype=np.float64
    )

    if (
        not np.all(
            np.isfinite(
                raw
            )
        )
        or
        raw[0] <= 0
        or
        raw[1] < 0
    ):

        raise RuntimeError(
            f"Invalid tau target: {raw}"
        )

    z = np.log1p(
        raw
    )

    return torch.tensor(
        z,
        dtype=torch.float32,
        device=device
    )


# =====================================================================================
# 9. TAIL-AWARE LOSS
# =====================================================================================

def batch_loss(
    records,
    ix,
    hnet,
    tnet
):

    LP = []
    LRHO = []
    LT = []

    for j in ix:

        r = records[
            int(j)
        ]

        p = torch.tensor(
            r.p,
            dtype=torch.float32,
            device=device
        )

        ph = phat_tensor(
            hnet,
            r
        )

        # PMF loss
        LP.append(
            torch.sum(
                (
                    ph - p
                )**2
            )
        )

        # Tail loss
        rho = tail_torch(
            p
        )

        rhoh = tail_torch(
            ph
        )

        LRHO.append(
            torch.mean(
                (
                    rhoh - rho
                )**2
            )
        )

        # Extinction-time loss
        if r.tau_valid:

            z = tau_target(
                r
            )

            zh = tnet(
                xtau(
                    r
                ).unsqueeze(0)
            ).squeeze(0)

            LT.append(
                torch.sum(
                    (
                        zh - z
                    )**2
                    /
                    (
                        1
                        +
                        z * z
                    )
                )
            )

    lp = torch.stack(
        LP
    ).mean()

    lrho = torch.stack(
        LRHO
    ).mean()

    lt = (
        torch.stack(
            LT
        ).mean()

        if LT

        else

        torch.zeros(
            (),
            device=device
        )
    )

    total = (
        lp
        +
        cfg.lambda_rho
        *
        lrho
        +
        cfg.lambda_tau
        *
        lt
    )

    return (
        total,
        lp,
        lrho,
        lt
    )


# =====================================================================================
# 10. TRAINER
# =====================================================================================

@torch.no_grad()
def validation_loss(
    hnet,
    tnet
):

    hnet.eval()
    tnet.eval()

    L, _, _, _ = batch_loss(
        valid,
        np.arange(
            len(valid)
        ),
        hnet,
        tnet
    )

    return L.item()


def train_emulator(
    records,
    seed,
    verbose=False
):

    seed_all(
        seed
    )

    hnet = HazardNet().to(
        device
    )

    tnet = TauNet().to(
        device
    )

    pars = (
        list(
            hnet.parameters()
        )
        +
        list(
            tnet.parameters()
        )
    )

    opt = torch.optim.AdamW(
        pars,
        lr=cfg.lr,
        weight_decay=
            cfg.weight_decay
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt,
            mode="min",
            factor=.5,
            patience=15
        )
    )

    rng = np.random.default_rng(
        seed + 7011
    )

    best = np.inf

    best_epoch = None

    bh = None
    bt = None

    wait = 0

    history = {
        "joint": [],
        "pmf": [],
        "tail": [],
        "tau": [],
        "validation": []
    }

    t0 = (
        time.perf_counter()
    )

    stopped_epoch = None

    for epoch in range(
        1,
        cfg.epochs + 1
    ):

        hnet.train()
        tnet.train()

        perm = rng.permutation(
            len(records)
        )

        J = []
        P = []
        RHO = []
        T = []

        for s in range(
            0,
            len(perm),
            cfg.batch_size
        ):

            ix = perm[
                s:
                s
                +
                cfg.batch_size
            ]

            opt.zero_grad()

            (
                L,
                Lp,
                Lrho,
                Lt

            ) = batch_loss(
                records,
                ix,
                hnet,
                tnet
            )

            if not torch.isfinite(
                L
            ):

                raise RuntimeError(
                    f"Non-finite loss: "
                    f"R={len(records)}, "
                    f"epoch={epoch}"
                )

            L.backward()

            torch.nn.utils.clip_grad_norm_(
                pars,
                cfg.grad_clip
            )

            opt.step()

            J.append(
                L.detach().item()
            )

            P.append(
                Lp.detach().item()
            )

            RHO.append(
                Lrho.detach().item()
            )

            T.append(
                Lt.detach().item()
            )

        j = np.mean(
            J
        )

        p = np.mean(
            P
        )

        rho = np.mean(
            RHO
        )

        tau = np.mean(
            T
        )

        v = validation_loss(
            hnet,
            tnet
        )

        scheduler.step(
            v
        )

        history[
            "joint"
        ].append(
            j
        )

        history[
            "pmf"
        ].append(
            p
        )

        history[
            "tail"
        ].append(
            rho
        )

        history[
            "tau"
        ].append(
            tau
        )

        history[
            "validation"
        ].append(
            v
        )

        if (
            bh is None
            or
            v
            <
            best
            -
            cfg.min_delta
        ):

            best = v

            best_epoch = (
                epoch
            )

            bh = copy.deepcopy(
                hnet.state_dict()
            )

            bt = copy.deepcopy(
                tnet.state_dict()
            )

            wait = 0

        else:

            wait += 1

        if (
            verbose
            and
            (
                epoch == 1
                or
                epoch % 20 == 0
            )
        ):

            print(
                f"R={len(records):5d} | "
                f"epoch={epoch:4d} | "
                f"joint={j:.3e} | "
                f"pmf={p:.3e} | "
                f"tail={rho:.3e} | "
                f"tau={tau:.3e} | "
                f"val={v:.3e} | "
                f"best_epoch={best_epoch}"
            )

        if (
            wait
            >=
            cfg.patience
        ):

            stopped_epoch = (
                epoch
            )

            print(
                f"Early stopping for R={len(records)}: "
                f"stop epoch={stopped_epoch}, "
                f"best epoch={best_epoch}, "
                f"best val={best:.4e}"
            )

            break

    if stopped_epoch is None:

        stopped_epoch = (
            cfg.epochs
        )

    hnet.load_state_dict(
        bh
    )

    tnet.load_state_dict(
        bt
    )

    return {

        "hazard":
            hnet,

        "tau":
            tnet,

        "history":
            history,

        "training_time":
            time.perf_counter()
            -
            t0,

        "best_validation_loss":
            best,

        "best_epoch":
            best_epoch,

        "stopped_epoch":
            stopped_epoch,

        "n_tau_train":
            sum(
                r.tau_valid
                for r in records
            )
    }


# =====================================================================================
# 11. EVALUATION
# =====================================================================================

def tail_np(p):

    return np.flip(
        np.cumsum(
            np.flip(
                p[1:]
            )
        )
    )


@torch.no_grad()
def predict(
    r,
    hnet,
    tnet
):

    hnet.eval()
    tnet.eval()

    p = phat_tensor(
        hnet,
        r
    ).cpu().numpy()

    z = (
        tnet(
            xtau(
                r
            ).unsqueeze(0)
        )
        .squeeze(0)
        .cpu()
        .numpy()
        .astype(
            np.float64
        )
    )

    moments = np.expm1(
        np.clip(
            z,
            0.,
            700.
        )
    )

    return (
        p,
        float(
            moments[0]
        ),
        float(
            moments[1]
        )
    )


def evaluate(
    records,
    hnet,
    tnet,
    split
):

    ans = []

    for j, r in enumerate(
        records
    ):

        p, m, v = predict(
            r,
            hnet,
            tnet
        )

        rho = tail_np(
            r.p
        )

        rhoh = tail_np(
            p
        )

        pos = (
            r.p
            >
            0
        )

        psafe = np.clip(
            p,
            cfg.kl_eps,
            1.
        )

        R0 = (
            r.beta
            /
            r.gamma
        )

        row = {

            "index":
                j,

            "split":
                split,

            "N":
                r.N,

            "i0":
                r.i0,

            "beta":
                r.beta,

            "gamma":
                r.gamma,

            "omega":
                r.omega,

            "R0":
                R0,

            "tau_valid":
                r.tau_valid,

            "E2":
                float(
                    np.linalg.norm(
                        p
                        -
                        r.p
                    )
                ),

            "E_rho":
                float(
                    np.max(
                        np.abs(
                            rhoh
                            -
                            rho
                        )
                    )
                ),

            "E_overflow":
                float(
                    abs(
                        p[-1]
                        -
                        r.p[-1]
                    )
                ),

            "KL":
                float(
                    np.sum(
                        r.p[pos]
                        *
                        np.log(
                            r.p[pos]
                            /
                            psafe[pos]
                        )
                    )
                ),

            "exact_p":
                r.p,

            "pred_p":
                p,

            "exact_tail":
                rho,

            "pred_tail":
                rhoh
        }

        if r.tau_valid:

            row[
                "mean_tau_relative_error"
            ] = float(
                abs(
                    m
                    -
                    r.mean_tau
                )
                /
                r.mean_tau
            )

            row[
                "var_tau_relative_error"
            ] = float(
                abs(
                    v
                    -
                    r.var_tau
                )
                /
                max(
                    r.var_tau,
                    1e-300
                )
            )

        else:

            row[
                "mean_tau_relative_error"
            ] = np.nan

            row[
                "var_tau_relative_error"
            ] = np.nan

        ans.append(
            row
        )

    return ans


METRICS = [
    "E2",
    "E_rho",
    "E_overflow",
    "KL",
    "mean_tau_relative_error",
    "var_tau_relative_error"
]


def finite_values(
    results,
    key
):

    x = np.asarray([
        r[key]
        for r in results
    ], dtype=float)

    return x[
        np.isfinite(
            x
        )
    ]


def median_metric(
    results,
    key
):

    x = finite_values(
        results,
        key
    )

    return (
        float(
            np.median(
                x
            )
        )

        if len(x)

        else

        np.nan
    )


# =====================================================================================
# 12. BALANCED NESTED TRAINING ORDER
# =====================================================================================

def balanced_order(
    records,
    seed
):

    rng = np.random.default_rng(
        seed
    )

    groups = {}

    for N in cfg.train_N:

        for flag in (
            0,
            1
        ):

            x = np.asarray([
                j
                for j, r in enumerate(
                    records
                )
                if
                r.N == N
                and
                int(
                    r.i0 == 1
                ) == flag
            ], dtype=int)

            rng.shuffle(
                x
            )

            groups[
                (N, flag)
            ] = list(
                x
            )

    ptr = {
        k: 0
        for k in groups
    }

    usedN = {
        N: 0
        for N in cfg.train_N
    }

    used1 = 0

    order = []

    for position in range(
        len(records)
    ):

        target1 = (
            cfg.i0_one_fraction
            *
            (
                position + 1
            )
        )

        preferred_flag = (
            1
            if
            used1 < target1
            else
            0
        )

        selected = None

        for flag in (
            preferred_flag,
            1 - preferred_flag
        ):

            candidates = [
                N
                for N in cfg.train_N
                if
                ptr[
                    (N, flag)
                ]
                <
                len(
                    groups[
                        (N, flag)
                    ]
                )
            ]

            if candidates:

                min_used = min(
                    usedN[
                        N
                    ]
                    for N in candidates
                )

                candidates = [
                    N
                    for N in candidates
                    if
                    usedN[N]
                    ==
                    min_used
                ]

                N = int(
                    rng.choice(
                        candidates
                    )
                )

                selected = (
                    N,
                    flag
                )

                break

        if selected is None:

            raise RuntimeError(
                "Could not construct balanced nested order."
            )

        N, flag = (
            selected
        )

        j = groups[
            (N, flag)
        ][
            ptr[
                (N, flag)
            ]
        ]

        ptr[
            (N, flag)
        ] += 1

        usedN[
            N
        ] += 1

        used1 += (
            flag
        )

        order.append(
            j
        )

    return np.asarray(
        order,
        dtype=int
    )


# =====================================================================================
# 13. FIXED SHOWCASE CONFIGURATIONS
#
# IMPORTANT FIX:
# No hard-coded N=200.
# Uses largest available exact-grid population size automatically.
# =====================================================================================

def entropy(p):

    z = p[
        p > 0
    ]

    return float(
        -np.sum(
            z
            *
            np.log(
                z
            )
        )
    )


def bimodality(p):

    z = (
        p[:-1]
    )

    if (
        len(z) < 3
        or
        z.max() <= 0
    ):

        return 0.

    peaks = [
        j
        for j in range(
            len(z)
        )
        if
        z[j]
        >=
        (
            z[j - 1]
            if j
            else
            -np.inf
        )
        and
        z[j]
        >=
        (
            z[j + 1]
            if
            j < len(z) - 1
            else
            -np.inf
        )
        and
        z[j]
        >=
        .03
        *
        z.max()
    ]

    if len(
        peaks
    ) < 2:

        return 0.

    a, b = sorted(
        peaks,
        key=
            lambda j:
                z[j],
        reverse=True
    )[:2]

    return float(
        min(
            z[a],
            z[b]
        )
        /
        max(
            z[a],
            z[b]
        )
        *
        abs(
            a - b
        )
        /
        max(
            len(z) - 1,
            1
        )
    )


# ------------------------------------------------------------
# Exact-grid showcase:
# automatically use largest exact N.
# ------------------------------------------------------------

SHOWCASE_N = max(
    EXACT_TEST_N
)


exact_large_i1 = [
    r
    for r in test_seen
    if
    r.N == SHOWCASE_N
    and
    r.i0 == 1
]


if exact_large_i1:

    case1 = max(
        exact_large_i1,
        key=
            lambda r:
                entropy(
                    r.p
                )
    )

else:

    exact_large = [
        r
        for r in test_seen
        if
        r.N == SHOWCASE_N
    ]

    if not exact_large:

        raise RuntimeError(
            f"No exact-test configurations available "
            f"at N={SHOWCASE_N}."
        )

    case1 = max(
        exact_large,
        key=
            lambda r:
                entropy(
                    r.p
                )
    )


# ------------------------------------------------------------
# Held-out interpolation showcase:
# strongest bimodality.
# ------------------------------------------------------------

if len(
    test_interp
) == 0:

    raise RuntimeError(
        "Interpolation test set is empty."
    )


case2 = max(
    test_interp,
    key=
        lambda r:
            bimodality(
                r.p
            )
)


SHOWCASES = [

    (
        rf"Exact-grid $N={SHOWCASE_N},\ i_0={case1.i0}$",
        case1
    ),

    (
        "Held-out interpolation",
        case2
    )
]


print(
    "\nFixed showcase configurations"
)

print(
    "-" * 70
)

print(
    "Showcase 1:",
    f"N={case1.N}, "
    f"i0={case1.i0}, "
    f"R0={case1.beta/case1.gamma:.3f}"
)

print(
    "Showcase 2:",
    f"N={case2.N}, "
    f"i0={case2.i0}, "
    f"R0={case2.beta/case2.gamma:.3f}"
)


# =====================================================================================
# 14. SINGLE-REPLICATION LEARNING-CURVE EXPERIMENT
# =====================================================================================

rows = []


gallery = {
    label: {}
    for label, _ in SHOWCASES
}


print(
    "\n"
    +
    "=" * 115
)

print(
    "SINGLE LEARNING-CURVE REPLICATION"
)

print(
    "=" * 115
)


order = balanced_order(
    train,
    cfg.seed + 20000
)


for R in R_VALUES:

    subset = [
        train[
            int(j)
        ]
        for j in order[
            :R
        ]
    ]

    counts = {
        N:
            sum(
                r.N == N
                for r in subset
            )
        for N in cfg.train_N
    }

    n_i1 = sum(
        r.i0 == 1
        for r in subset
    )

    n_tau = sum(
        r.tau_valid
        for r in subset
    )

    print(
        "\n"
        +
        "-" * 115
    )

    print(
        f"R={R}"
    )

    print(
        "N counts:",
        counts
    )

    print(
        f"i0=1: "
        f"{n_i1}/{R} "
        f"({n_i1/R:.1%}) | "
        f"valid tau: "
        f"{n_tau}/{R}"
    )

    fit = train_emulator(
        subset,
        seed=
            cfg.seed
            +
            30000
            +
            R,
        verbose=True
    )

    hnet = fit[
        "hazard"
    ]

    tnet = fit[
        "tau"
    ]

    exact = evaluate(
        test_seen,
        hnet,
        tnet,
        "exact"
    )

    interp = evaluate(
        test_interp,
        hnet,
        tnet,
        "interpolation"
    )

    interp_i1 = [
        x
        for x in interp
        if
        x["i0"] == 1
    ]

    interp_i_gt1 = [
        x
        for x in interp
        if
        x["i0"] > 1
    ]

    for split, res in [

        (
            "exact",
            exact
        ),

        (
            "interpolation",
            interp
        ),

        (
            "interpolation_i0_1",
            interp_i1
        ),

        (
            "interpolation_i0_gt1",
            interp_i_gt1
        )
    ]:

        row = {

            "R":
                R,

            "split":
                split,

            "training_time":
                fit[
                    "training_time"
                ],

            "best_validation_loss":
                fit[
                    "best_validation_loss"
                ],

            "best_epoch":
                fit[
                    "best_epoch"
                ],

            "stopped_epoch":
                fit[
                    "stopped_epoch"
                ],

            "n_tau_train":
                fit[
                    "n_tau_train"
                ],

            "n_tau_eval":
                sum(
                    x[
                        "tau_valid"
                    ]
                    for x in res
                )
        }

        for key in METRICS:

            row[
                key
            ] = median_metric(
                res,
                key
            )

        rows.append(
            row
        )

    if R in GALLERY_R:

        for label, r in SHOWCASES:

            gallery[
                label
            ][
                R
            ] = evaluate(
                [r],
                hnet,
                tnet,
                "gallery"
            )[0]

    print(
        f"interp: "
        f"E2={median_metric(interp,'E2'):.4e} | "
        f"E_rho={median_metric(interp,'E_rho'):.4e} | "
        f"E(tau)="
        f"{median_metric(interp,'mean_tau_relative_error'):.4e} | "
        f"Var(tau)="
        f"{median_metric(interp,'var_tau_relative_error'):.4e}"
    )

    print(
        f"best epoch="
        f"{fit['best_epoch']} | "
        f"stop epoch="
        f"{fit['stopped_epoch']} | "
        f"best val="
        f"{fit['best_validation_loss']:.4e}"
    )

    del (
        fit,
        hnet,
        tnet,
        exact,
        interp,
        interp_i1,
        interp_i_gt1
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =====================================================================================
# 15. SINGLE-REPLICATION CURVES
# =====================================================================================

def curve(
    split,
    key
):

    values = []

    for R in R_VALUES:

        matches = [
            z[
                key
            ]
            for z in rows
            if
            z[
                "R"
            ] == R
            and
            z[
                "split"
            ] == split
        ]

        values.append(
            matches[0]
            if matches
            else
            np.nan
        )

    return (
        np.asarray(
            R_VALUES
        ),

        np.asarray(
            values,
            dtype=float
        )
    )


print(
    "\n"
    +
    "=" * 120
)

print(
    "HELD-OUT POPULATION-SIZE INTERPOLATION SUMMARY"
)

print(
    "=" * 120
)


print(
    f"{'R':>7s} | "
    f"{'E2':>11s} | "
    f"{'E_rho':>11s} | "
    f"{'E_over':>11s} | "
    f"{'KL':>11s} | "
    f"{'Rel E(tau)':>12s} | "
    f"{'Rel Var(tau)':>14s} | "
    f"{'Best ep.':>8s} | "
    f"{'Stop ep.':>8s}"
)

print(
    "-" * 135
)


for R in R_VALUES:

    row = [
        z
        for z in rows
        if
        z["R"] == R
        and
        z["split"]
        ==
        "interpolation"
    ][0]

    print(
        f"{R:7d} | "
        f"{row['E2']:11.4e} | "
        f"{row['E_rho']:11.4e} | "
        f"{row['E_overflow']:11.4e} | "
        f"{row['KL']:11.4e} | "
        f"{row['mean_tau_relative_error']:12.4e} | "
        f"{row['var_tau_relative_error']:14.4e} | "
        f"{row['best_epoch']:8d} | "
        f"{row['stopped_epoch']:8d}"
    )


# =====================================================================================
# 16. FIGURE 1 — EXACT VS INTERPOLATION
# =====================================================================================

plt.rcParams.update({
    "font.size":
        10.5,

    "axes.spines.top":
        False,

    "axes.spines.right":
        False
})


fig, axs = plt.subplots(
    2,
    2,
    figsize=(13, 9)
)


spec = [

    (
        "E2",
        r"Median $E_2$",
        "(A) Distributional error"
    ),

    (
        "E_rho",
        r"Median $E_\rho$",
        "(B) Tail-risk error"
    ),

    (
        "mean_tau_relative_error",
        "Median relative error",
        r"(C) $E(\tau)$"
    ),

    (
        "var_tau_relative_error",
        "Median relative error",
        r"(D) $\mathrm{Var}(\tau)$"
    )
]


for ax, (
    key,
    ylabel,
    title

) in zip(
    axs.flat,
    spec
):

    for (
        split,
        label,
        color,
        marker

    ) in [

        (
            "exact",
            "Exact-grid $N$",
            "#555555",
            "o"
        ),

        (
            "interpolation",
            "Interpolated $N$",
            "#0072B2",
            "D"
        )
    ]:

        R, m = curve(
            split,
            key
        )

        ok = np.isfinite(
            m
        )

        ax.plot(
            R[
                ok
            ],

            np.maximum(
                m[
                    ok
                ],
                1e-12
            ),

            color=color,
            marker=marker,
            lw=2,
            label=label
        )

    ax.set_xscale(
        "log"
    )

    ax.set_yscale(
        "log"
    )

    ax.set_xticks(
        R_VALUES
    )

    ax.set_xticklabels([
        str(R)
        for R in R_VALUES
    ])

    ax.set_xlabel(
        r"Exact training configurations $R$"
    )

    ax.set_ylabel(
        ylabel
    )

    ax.set_title(
        title
    )

    ax.grid(
        alpha=.15
    )

    ax.legend(
        frameon=False
    )


fig.suptitle(
    "Accuracy Versus Exact-Teacher Training-Set Size",
    fontsize=15
)


plt.tight_layout()


plt.savefig(
    out
    /
    "figure_5_1B_learning_curves_single_rep.png",
    dpi=cfg.dpi,
    bbox_inches="tight"
)


plt.show()


# =====================================================================================
# 17. FIGURE 2 — i0=1 VS i0>1
# =====================================================================================

fig, axs = plt.subplots(
    2,
    3,
    figsize=(17, 9)
)


spec2 = [

    (
        "E2",
        r"$E_2$",
        "(A) Distributional error"
    ),

    (
        "E_rho",
        r"$E_\rho$",
        "(B) Tail-risk error"
    ),

    (
        "E_overflow",
        "Overflow error",
        "(C) Overflow risk"
    ),

    (
        "KL",
        "KL divergence",
        "(D) KL divergence"
    ),

    (
        "mean_tau_relative_error",
        "Relative error",
        r"(E) $E(\tau)$"
    ),

    (
        "var_tau_relative_error",
        "Relative error",
        r"(F) $\mathrm{Var}(\tau)$"
    )
]


for ax, (
    key,
    ylabel,
    title

) in zip(
    axs.flat,
    spec2
):

    for (
        split,
        label,
        color,
        marker

    ) in [

        (
            "interpolation_i0_1",
            r"$i_0=1$",
            "#0072B2",
            "o"
        ),

        (
            "interpolation_i0_gt1",
            r"$i_0>1$",
            "#E69F00",
            "s"
        )
    ]:

        R, m = curve(
            split,
            key
        )

        ok = np.isfinite(
            m
        )

        ax.plot(
            R[
                ok
            ],

            np.maximum(
                m[
                    ok
                ],
                1e-12
            ),

            color=color,
            marker=marker,
            lw=2,
            label=label
        )

    ax.set_xscale(
        "log"
    )

    ax.set_yscale(
        "log"
    )

    ax.set_xticks(
        R_VALUES
    )

    ax.set_xticklabels([
        str(R)
        for R in R_VALUES
    ])

    ax.set_xlabel(
        r"Exact training configurations $R$"
    )

    ax.set_ylabel(
        ylabel
    )

    ax.set_title(
        title
    )

    ax.grid(
        alpha=.15
    )

    ax.legend(
        frameon=False
    )


fig.suptitle(
    r"Interpolation Accuracy for $i_0=1$ and $i_0>1$",
    fontsize=15
)


plt.tight_layout()


plt.savefig(
    out
    /
    "figure_5_1B_i0_learning_curves_single_rep.png",
    dpi=cfg.dpi,
    bbox_inches="tight"
)


plt.show()


# =====================================================================================
# 18. FIGURE 3 — PMF RECONSTRUCTION AS R INCREASES
# =====================================================================================

fig, axs = plt.subplots(
    2,
    len(
        GALLERY_R
    ),
    figsize=(19, 8),
    squeeze=False
)


for row, (
    label,
    _

) in enumerate(
    SHOWCASES
):

    for col, R in enumerate(
        GALLERY_R
    ):

        ax = axs[
            row,
            col
        ]

        z = gallery[
            label
        ][
            R
        ]

        c = np.arange(
            z[
                "N"
            ]
            +
            1
        )

        ax.bar(
            c,
            z[
                "exact_p"
            ][:-1],
            width=.85,
            color=".82",
            label=
                "Exact Markovian"
        )

        ax.plot(
            c,
            z[
                "pred_p"
            ][:-1],
            color="#D55E00",
            lw=1.8,
            label=
                "Neural emulator"
        )

        ax.set_title(
            f"{label}, $R={R}$\n"
            f"$N={z['N']}$, "
            f"$i_0={z['i0']}$, "
            rf"$R_0={z['R0']:.2f}$, "
            rf"$E_2={z['E2']:.3f}$, "
            rf"$E_\rho={z['E_rho']:.3f}$"
        )

        ax.set_xlabel(
            "Infection count $c$"
        )

        ax.set_ylabel(
            "Probability mass"
        )

        if (
            row == 0
            and
            col == 0
        ):

            ax.legend(
                frameon=False
            )


plt.tight_layout()


plt.savefig(
    out
    /
    "figure_5_1B_pmf_vs_R_single_rep.png",
    dpi=cfg.dpi,
    bbox_inches="tight"
)


plt.show()


# =====================================================================================
# 19. FIGURE 4 — TAIL RECONSTRUCTION AS R INCREASES
# =====================================================================================

fig, axs = plt.subplots(
    2,
    len(
        GALLERY_R
    ),
    figsize=(19, 8),
    squeeze=False
)


for row, (
    label,
    _

) in enumerate(
    SHOWCASES
):

    for col, R in enumerate(
        GALLERY_R
    ):

        ax = axs[
            row,
            col
        ]

        z = gallery[
            label
        ][
            R
        ]

        c = np.arange(
            z[
                "N"
            ]
            +
            1
        )

        ax.plot(
            c,
            z[
                "exact_tail"
            ],
            color="black",
            lw=2,
            label=
                "Exact Markovian"
        )

        ax.plot(
            c,
            z[
                "pred_tail"
            ],
            "--",
            color="#0072B2",
            lw=1.8,
            label=
                "Neural emulator"
        )

        ax.set_ylim(
            -.01,
            1.01
        )

        ax.set_title(
            f"{label}, $R={R}$\n"
            f"$N={z['N']}$, "
            f"$i_0={z['i0']}$, "
            rf"$R_0={z['R0']:.2f}$, "
            rf"$E_\rho={z['E_rho']:.3f}$"
        )

        ax.set_xlabel(
            "Threshold $c$"
        )

        ax.set_ylabel(
            r"$P(C>c)$"
        )

        if (
            row == 0
            and
            col == 0
        ):

            ax.legend(
                frameon=False
            )


plt.tight_layout()


plt.savefig(
    out
    /
    "figure_5_1B_tail_vs_R_single_rep.png",
    dpi=cfg.dpi,
    bbox_inches="tight"
)


plt.show()


# =====================================================================================
# 20. SAVE
# =====================================================================================

result_path = (
    out
    /
    "section_5_1B_single_rep_results.pkl"
)


with open(
    result_path,
    "wb"
) as f:

    pickle.dump(
        {

            "config":
                asdict(
                    cfg
                ),

            "R_VALUES":
                R_VALUES,

            "REPLICATIONS":
                1,

            "EXACT_TEST_N":
                EXACT_TEST_N,

            "INTERP_TEST_N":
                INTERP_TEST_N,

            "rows":
                rows,

            "gallery":
                gallery,

            "tau_resolution": {

                "train":
                    (
                        sum(
                            r.tau_valid
                            for r in train
                        ),
                        len(
                            train
                        )
                    ),

                "validation":
                    (
                        sum(
                            r.tau_valid
                            for r in valid
                        ),
                        len(
                            valid
                        )
                    ),

                "test_exact":
                    (
                        sum(
                            r.tau_valid
                            for r in test_seen
                        ),
                        len(
                            test_seen
                        )
                    ),

                "test_interp":
                    (
                        sum(
                            r.tau_valid
                            for r in test_interp
                        ),
                        len(
                            test_interp
                        )
                    )
            }
        },

        f
    )


print(
    "\n"
    +
    "=" * 115
)

print(
    "EXPERIMENT 5.1-B COMPLETE — SINGLE REPLICATION"
)

print(
    "=" * 115
)


print(
    "Training N:",
    cfg.train_N
)

print(
    "Exact test N:",
    EXACT_TEST_N
)

print(
    "Interpolated test N:",
    INTERP_TEST_N
)

print(
    "R values:",
    R_VALUES
)

print(
    "Replications: 1"
)

print(
    "Tail-loss weight lambda_rho =",
    cfg.lambda_rho
)

print(
    "R0 = beta/gamma"
)

print(
    "No inverse of T or D0 was computed."
)


print(
    "\nResolved tau targets — train:",
    f"{sum(r.tau_valid for r in train)}/{len(train)}"
)

print(
    "Resolved tau targets — validation:",
    f"{sum(r.tau_valid for r in valid)}/{len(valid)}"
)

print(
    "Resolved tau targets — exact test:",
    f"{sum(r.tau_valid for r in test_seen)}/{len(test_seen)}"
)

print(
    "Resolved tau targets — interpolation test:",
    f"{sum(r.tau_valid for r in test_interp)}/{len(test_interp)}"
)


print(
    "\nResults:",
    result_path.resolve()
)

print(
    "=" * 115
)

EXPERIMENT 5.1-B — SINGLE-REPLICATION LEARNING CURVE
Device: cpu
Training N: (30, 50, 70, 90, 110, 130, 150)
Exact test N: (30, 50, 70, 90, 110, 130, 150)
Interpolated test N: (40, 60, 80, 100, 120, 140)
R values: (100, 200, 500, 1000, 2000, 5000, 10000, 20000, 30000, 50000)
Replications: 1
R0 = beta/gamma
No matrix inverse is computed.
Loading results_section_5_1B_R50000_single_rep/train_B_R50000_N150_tailaware_single_rep_v14_1fb659400fb5.pkl
Loading results_section_5_1B_R50000_single_rep/validation_B_R50000_N150_tailaware_single_rep_v14_1fb659400fb5.pkl
Loading results_section_5_1B_R50000_single_rep/test_exact_B_R50000_N150_tailaware_single_rep_v14_1fb659400fb5.pkl
Loading results_section_5_1B_R50000_single_rep/test_interp_B_R50000_N150_tailaware_single_rep_v14_1fb659400fb5.pkl

EXACT-TARGET AUDIT

TRAIN
N= 30 | n=7143 | i0=1=1786 | valid tau=7143/7143
N= 50 | n=7143 | i0=1=1786 | valid tau=7143/7143
N= 70 | n=7143 | i0=1=1786 | valid tau=7143/7143
N= 90 | n=7143 | i0=1=1786 | valid 